#### IMPORTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import os
import requests
import zipfile
from io import BytesIO
import re

In [2]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


#### DOWNLOAD AND PREP DATASET

In [3]:
def download_data():
    url = "https://download.pytorch.org/tutorial/data.zip"
    if not os.path.exists("data/eng-fra.txt"):
        print("!!! dataset not found")
        print("--- downloading dataset")
        req = requests.get(url)
        with zipfile.ZipFile(BytesIO(req.content)) as zip_ref:
            zip_ref.extractall(".")
    print("--- dataset ready")

download_data()

!!! dataset not found
--- downloading dataset
--- dataset ready


In [4]:
# preping data for tokens

SOS_token = 0 # start of sentence
EOS_token = 1 # end of sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2 

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [5]:
def normalizeString(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

s = "Hello! How are you? I'm fine."
print(normalizeString(s))

hello ! how are you ? i m fine .


In [6]:
print("--- reading lines")
lines = open('data/eng-fra.txt', encoding='utf-8').read().strip().split('\n')
pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

# filter for short sentences for quick training
MAX_LENGTH = 10
good_prefixes = ("i am ", "i m ", "he is", "he s ", "she is", "she s ", "you are", "you re ", "we are", "we re ", "they are", "they re ")
pairs = [p for p in pairs if len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH and p[0].startswith(good_prefixes)]

input_lang = Lang('eng')
output_lang = Lang('fra')

print("--- building vocab")
for pair in pairs:
    input_lang.addSentence(pair[0])
    output_lang.addSentence(pair[1])

print(f"+++ counted words: \neng: {input_lang.n_words}\nfra: {output_lang.n_words}")

--- reading lines
--- building vocab
+++ counted words: 
eng: 2728
fra: 3899


#### BUILDING MODEL ARCHITECTURE

In [7]:
class encoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(encoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input_seq):
        # input_seq shape: (batch_size, seq_len)
        embedded = self.embedding(input_seq)
        
        # output contains the hidden states for all timesteps
        # hidden contains the final context vector
        output, hidden = self.gru(embedded)
        return output, hidden

class decoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(decoderRNN, self).__init__()
        self.hidden_size = hidden_size
        
        self.embedding = nn.Embedding(output_size, hidden_size)
        
        # input to the GRU: the embedded word + context vector
        self.gru = nn.GRU(hidden_size * 2, hidden_size, batch_first=True)

        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input_step, hidden, context):
        # input_step shape: (batch_size, 1) - one word at a time
        embedded = self.embedding(input_step)

        # reshape context to (batch_size, 1, hidden_size) from (1, batch_size, hidden_size)
        context_reshaped = context.permute(1, 0, 2)
        
        # concatenate embedding and context vector (batch_size, 1, 2*hidden_size)
        emb_con = torch.cat((embedded, context_reshaped), dim=2)
        
        output, hidden = self.gru(emb_con, hidden)
        
        # decode next word
        prediction = self.out(output.squeeze(1))
        return prediction, hidden

In [14]:
def tensorFromSentence(lang, sentence):
    indexes = [lang.word2index[word] for word in sentence.split(' ')]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

s = "i m fine"
print(tensorFromSentence(input_lang, s))

tensor([[ 2,  3, 23,  1]])


In [15]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    # encode eng sequence
    encoder_outputs, encoder_hidden = encoder(input_tensor)
    
    # the context vector
    context = encoder_hidden

    # decoder starts with SOS token and the context vector as its initial hidden state
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = context 

    loss = 0
    target_length = target_tensor.size(1)

    # teacher forcing: feed the target as the next input instead of decoder's own prediction
    for di in range(target_length):
        decoder_output, decoder_hidden = decoder(
            decoder_input, decoder_hidden, context
        )
        loss += criterion(decoder_output, target_tensor[0, di].unsqueeze(0))
        decoder_input = target_tensor[:, di].unsqueeze(1) # next target word

    loss.backward()
    
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item()/target_length

In [18]:
# init models
hidden_size = 256
encoder = encoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = decoderRNN(hidden_size, output_lang.n_words).to(device)

# set optim, loss function and lr
encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# training loop
epochs = 1000 
print("--- starting training")
for epoch in range(1, epochs + 1):
    training_pair = random.choice(pairs)
    input_tensor = tensorFromSentence(input_lang, training_pair[0])
    target_tensor = tensorFromSentence(output_lang, training_pair[1])

    loss = train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
    
    if epoch % (epochs // 10) == 0:
        print(f"+++ epoch {epoch}/{epochs} | loss: {loss:.4f}")

--- starting training
+++ epoch 100/1000 | loss: 4.1167
+++ epoch 200/1000 | loss: 5.2720
+++ epoch 300/1000 | loss: 4.1913
+++ epoch 400/1000 | loss: 2.7980
+++ epoch 500/1000 | loss: 3.6979
+++ epoch 600/1000 | loss: 4.2211
+++ epoch 700/1000 | loss: 2.2525
+++ epoch 800/1000 | loss: 3.9031
+++ epoch 900/1000 | loss: 4.3961
+++ epoch 1000/1000 | loss: 2.4520


In [ ]:
def evaluate(encoder, decoder, sentence):
    # no grad as we are not training
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        
        _, encoder_hidden = encoder(input_tensor)
        context = encoder_hidden
        
        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = context
        
        decoded_words = []

        for di in range(MAX_LENGTH):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, context)
            
            topv, topi = decoder_output.data.topk(1)
            # break loop if EOS token is predicted
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
                
            decoder_input = topi.view(1, 1)

        return ' '.join(decoded_words)

In [20]:
print("---translating")
for i in range(5):
    pair = random.choice(pairs)
    print(">", pair[0])
    print("=", pair[1])
    output_sentence = evaluate(encoder, decoder, pair[0])
    print("<", output_sentence)
    print("")

---translating
> we re both witnesses .
= nous sommes toutes deux t moins .
< nous sommes en train de la maison . <EOS>

> she is already married .
= elle est d j mari e .
< elle est en as . <EOS>

> i m your partner .
= je suis ton partenaire .
< je suis heureux . <EOS>

> you re not that interesting .
= tu n es pas si int ressant .
< tu n es pas si ? <EOS>

> she is connected with that company .
= elle est en relation avec cette soci t .
< elle est en as de partir . <EOS>

